# SOPR Double Capitulation Signal Analysis

**Hypothesis:** When both STH SOPR < 1 AND Entity-Adjusted SOPR < 1, it's a strong buy signal.

**Logic:** 
- SOPR < 1 = coins being spent at a loss (capitulation)
- STH SOPR < 1 = short-term holders selling at a loss
- Both < 1 = broad capitulation = local bottom

## Contents
1. Data Loading & Exploration
2. Regression Analysis
3. Cycle-by-Cycle Consistency
4. Signal Generation
5. Backtest
6. Walk-Forward Validation

---
## 1. Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt
import statsmodels.api as sm
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# VectorBT settings
vbt.settings.plotting['use_widgets'] = False
vbt.settings.array_wrapper['freq'] = 'D'
vbt.settings.portfolio['init_cash'] = 100_000

print(f"VectorBT version: {vbt.__version__}")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

# Load SOPR metrics
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

# Merge
df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()

# Filter to 2019+
START_DATE = '2018-12-15'
df = df[df.index >= START_DATE]

print(f"Data loaded: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"\nSOPR range: {df['sopr'].min():.4f} - {df['sopr'].max():.4f}")
print(f"STH SOPR range: {df['sopr_sth'].min():.4f} - {df['sopr_sth'].max():.4f}")

In [ ]:
# Create forward returns
FORWARD_DAYS = 30
df['fwd_return'] = df['price'].pct_change(FORWARD_DAYS).shift(-FORWARD_DAYS)

# Create signal conditions
df['sopr_below_1'] = df['sopr'] < 1
df['sopr_sth_below_1'] = df['sopr_sth'] < 1
df['double_capitulation'] = df['sopr_below_1'] & df['sopr_sth_below_1']

print(f"\nSignal Frequencies:")
print(f"  SOPR < 1: {df['sopr_below_1'].sum()} days ({df['sopr_below_1'].mean()*100:.1f}%)")
print(f"  STH SOPR < 1: {df['sopr_sth_below_1'].sum()} days ({df['sopr_sth_below_1'].mean()*100:.1f}%)")
print(f"  BOTH < 1: {df['double_capitulation'].sum()} days ({df['double_capitulation'].mean()*100:.1f}%)")

In [ ]:
# Visualize
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['Bitcoin Price', 'SOPR (Entity-Adjusted)', 'STH SOPR'],
                    row_heights=[0.4, 0.3, 0.3])

# Price with signal markers
fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price', line=dict(color='blue')), row=1, col=1)

# Mark double capitulation days
cap_days = df[df['double_capitulation']]
fig.add_trace(go.Scatter(x=cap_days.index, y=cap_days['price'], mode='markers',
                         marker=dict(color='green', size=6, symbol='circle'),
                         name='Double Capitulation'), row=1, col=1)

# SOPR
fig.add_trace(go.Scatter(x=df.index, y=df['sopr'], name='SOPR', line=dict(color='orange')), row=2, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=2, col=1)

# STH SOPR
fig.add_trace(go.Scatter(x=df.index, y=df['sopr_sth'], name='STH SOPR', line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=3, col=1)

fig.update_layout(height=800, showlegend=True)
fig.show()

---
## 2. Regression Analysis

Does the double capitulation signal predict forward returns?

In [ ]:
# Regression: fwd_return ~ double_capitulation
reg_data = df[['double_capitulation', 'fwd_return']].dropna()

X = reg_data['double_capitulation'].astype(float)
Y = reg_data['fwd_return']
X_const = sm.add_constant(X)

model = sm.OLS(Y, X_const).fit()
print(model.summary())

In [ ]:
# Interpret results
coef = model.params['double_capitulation']
pval = model.pvalues['double_capitulation']
r2 = model.rsquared

print("\n" + "="*60)
print("REGRESSION RESULTS")
print("="*60)
print(f"\nCoefficient: {coef:.4f} ({coef*100:.2f}%)")
print(f"p-value: {pval:.6f}")
print(f"R-squared: {r2:.4f}")

print(f"\nInterpretation:")
print(f"  When double capitulation occurs, {FORWARD_DAYS}-day returns are")
print(f"  on average {coef*100:+.2f}% {'higher' if coef > 0 else 'lower'} than normal.")

if pval < 0.05:
    print(f"\n  ✓ STATISTICALLY SIGNIFICANT (p < 0.05)")
else:
    print(f"\n  ✗ NOT statistically significant (p >= 0.05)")

In [ ]:
# Compare returns: signal days vs non-signal days
signal_returns = df[df['double_capitulation']]['fwd_return'].dropna()
no_signal_returns = df[~df['double_capitulation']]['fwd_return'].dropna()

print("\n" + "="*60)
print("RETURN COMPARISON")
print("="*60)
print(f"\n{'Metric':<25} {'Signal Days':>15} {'Other Days':>15}")
print("-"*60)
print(f"{'Count':<25} {len(signal_returns):>15} {len(no_signal_returns):>15}")
print(f"{'Mean Return':<25} {signal_returns.mean()*100:>14.2f}% {no_signal_returns.mean()*100:>14.2f}%")
print(f"{'Median Return':<25} {signal_returns.median()*100:>14.2f}% {no_signal_returns.median()*100:>14.2f}%")
print(f"{'Std Dev':<25} {signal_returns.std()*100:>14.2f}% {no_signal_returns.std()*100:>14.2f}%")
print(f"{'Win Rate (>0)':<25} {(signal_returns > 0).mean()*100:>14.1f}% {(no_signal_returns > 0).mean()*100:>14.1f}%")
print(f"{'Sharpe (annualized)':<25} {signal_returns.mean()/signal_returns.std()*np.sqrt(12):>15.2f} {no_signal_returns.mean()/no_signal_returns.std()*np.sqrt(12):>15.2f}")

In [ ]:
# Visualize return distributions
fig = make_subplots(rows=1, cols=2, subplot_titles=['Signal Days Returns', 'All Days Returns'])

fig.add_trace(go.Histogram(x=signal_returns*100, nbinsx=30, name='Signal', 
                           marker_color='green', opacity=0.7), row=1, col=1)
fig.add_trace(go.Histogram(x=no_signal_returns*100, nbinsx=30, name='No Signal',
                           marker_color='gray', opacity=0.7), row=1, col=2)

fig.add_vline(x=0, line_dash='dash', line_color='red', row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='red', row=1, col=2)
fig.add_vline(x=signal_returns.mean()*100, line_color='green', row=1, col=1)
fig.add_vline(x=no_signal_returns.mean()*100, line_color='gray', row=1, col=2)

fig.update_layout(height=400, showlegend=False,
                  title_text=f'{FORWARD_DAYS}-Day Forward Returns Distribution')
fig.update_xaxes(title_text='Return (%)')
fig.show()

---
## 3. Cycle-by-Cycle Consistency

In [ ]:
# Define cycles
cycles = [
    {"name": "2019", "start": "2018-12-15", "end": "2019-06-26"},
    {"name": "2020-2021", "start": "2020-03-13", "end": "2021-11-10"},
    {"name": "2023-2024", "start": "2022-11-21", "end": "2024-03-14"},
    {"name": "2024-Present", "start": "2024-09-01", "end": "2026-12-31"},
]

cycle_results = []

for cycle in cycles:
    mask = (df.index >= cycle['start']) & (df.index < cycle['end'])
    cycle_data = df.loc[mask, ['double_capitulation', 'fwd_return']].dropna()
    
    if len(cycle_data) < 30 or cycle_data['double_capitulation'].sum() < 5:
        continue
    
    # Regression
    X = cycle_data['double_capitulation'].astype(float)
    Y = cycle_data['fwd_return']
    X_const = sm.add_constant(X)
    
    try:
        model_cycle = sm.OLS(Y, X_const).fit()
        
        # Also get simple stats
        sig_ret = cycle_data[cycle_data['double_capitulation']]['fwd_return']
        
        cycle_results.append({
            'cycle': cycle['name'],
            'n_days': len(cycle_data),
            'n_signals': cycle_data['double_capitulation'].sum(),
            'coefficient': model_cycle.params['double_capitulation'],
            'p_value': model_cycle.pvalues['double_capitulation'],
            'mean_signal_return': sig_ret.mean(),
            'win_rate': (sig_ret > 0).mean()
        })
    except:
        continue

cycle_df = pd.DataFrame(cycle_results)
print("CYCLE-BY-CYCLE RESULTS")
print("="*80)
print(cycle_df.to_string(index=False))

In [ ]:
# Consistency check
if len(cycle_df) > 0:
    n_positive = (cycle_df['coefficient'] > 0).sum()
    n_significant = (cycle_df['p_value'] < 0.1).sum()
    
    print(f"\n" + "="*60)
    print("CONSISTENCY CHECK")
    print("="*60)
    print(f"Positive coefficient: {n_positive}/{len(cycle_df)} cycles ({n_positive/len(cycle_df)*100:.0f}%)")
    print(f"Significant (p<0.1): {n_significant}/{len(cycle_df)} cycles ({n_significant/len(cycle_df)*100:.0f}%)")
    
    if n_positive == len(cycle_df):
        print("\n✓ CONSISTENT: Positive in ALL cycles!")
    elif n_positive >= len(cycle_df) * 0.6:
        print(f"\n✓ MOSTLY CONSISTENT: Positive in {n_positive}/{len(cycle_df)} cycles")
    else:
        print(f"\n✗ INCONSISTENT: Sign flips between cycles")

---
## 4. Signal Generation

Entry when BOTH SOPR < 1 and STH SOPR < 1

In [ ]:
# Prepare for backtest
close = df['price']

# Signal: Entry when entering double capitulation, Exit when leaving
in_signal = df['double_capitulation']

# Entry = transition into capitulation
entries = in_signal & ~in_signal.shift(1).fillna(False)

# Exit = transition out of capitulation
exits = ~in_signal & in_signal.shift(1).fillna(False)

print(f"Signal: Buy when SOPR < 1 AND STH SOPR < 1")
print(f"\nEntry signals: {entries.sum()}")
print(f"Exit signals: {exits.sum()}")
print(f"Days in signal: {in_signal.sum()} ({in_signal.mean()*100:.1f}%)")

In [ ]:
# Visualize entry points
fig = go.Figure()

fig.add_trace(go.Scatter(x=close.index, y=close, name='Price', line=dict(color='blue')))

# Entry markers
entry_prices = close[entries]
fig.add_trace(go.Scatter(x=entry_prices.index, y=entry_prices, mode='markers',
                         marker=dict(symbol='triangle-up', size=12, color='green'),
                         name='Entry (Capitulation Start)'))

# Exit markers
exit_prices = close[exits]
fig.add_trace(go.Scatter(x=exit_prices.index, y=exit_prices, mode='markers',
                         marker=dict(symbol='triangle-down', size=12, color='red'),
                         name='Exit (Capitulation End)'))

fig.update_layout(title='Double Capitulation Entry/Exit Points',
                  height=500, yaxis_type='log')
fig.show()

---
## 5. Backtest

In [ ]:
# Run backtest
pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    init_cash=100_000,
    fees=0.001,
    slippage=0.001,
    freq='D'
)

# Benchmark
pf_hold = vbt.Portfolio.from_holding(close, init_cash=100_000, freq='D')

print("STRATEGY STATS")
print("="*50)
print(pf.stats())

In [ ]:
# Compare to Buy & Hold
print("\n" + "="*60)
print("STRATEGY vs BUY & HOLD")
print("="*60)
print(f"{'Metric':<25} {'Strategy':>15} {'Buy & Hold':>15}")
print("-"*60)
print(f"{'Total Return':<25} {pf.total_return()*100:>14.1f}% {pf_hold.total_return()*100:>14.1f}%")
print(f"{'Sharpe Ratio':<25} {pf.sharpe_ratio():>15.2f} {pf_hold.sharpe_ratio():>15.2f}")
print(f"{'Sortino Ratio':<25} {pf.sortino_ratio():>15.2f} {pf_hold.sortino_ratio():>15.2f}")
print(f"{'Max Drawdown':<25} {pf.max_drawdown()*100:>14.1f}% {pf_hold.max_drawdown()*100:>14.1f}%")
print(f"{'Win Rate':<25} {pf.trades.win_rate()*100:>14.1f}% {'N/A':>15}")
print(f"{'Total Trades':<25} {pf.trades.count():>15} {'1':>15}")

excess = pf.total_return() - pf_hold.total_return()
print(f"\n{'Excess Return':<25} {excess*100:>+14.1f}%")

In [ ]:
# Equity curves
fig = go.Figure()

fig.add_trace(go.Scatter(x=pf.value().index, y=pf.value(), name='Double Capitulation Strategy'))
fig.add_trace(go.Scatter(x=pf_hold.value().index, y=pf_hold.value(), name='Buy & Hold'))

fig.update_layout(title='Equity Curves: Strategy vs Buy & Hold',
                  yaxis_title='Portfolio Value ($)', height=500)
fig.show()

In [ ]:
# Trade analysis
if pf.trades.count() > 0:
    print("\nTRADE DETAILS")
    print("="*60)
    trades = pf.trades.records_readable
    print(trades[['Entry Timestamp', 'Exit Timestamp', 'PnL', 'Return', 'Duration']].to_string())

---
## 6. Walk-Forward Validation

In [ ]:
# Walk-forward parameters
TRAIN_DAYS = 365
TEST_DAYS = 90
STEP_DAYS = 90

# For this signal, there's no threshold to optimize
# So we just test if the signal works in each period

wf_results = []

total_days = len(close)
n_folds = (total_days - TRAIN_DAYS) // STEP_DAYS

for fold in range(n_folds):
    test_start = TRAIN_DAYS + fold * STEP_DAYS
    test_end = min(test_start + TEST_DAYS, total_days)
    
    if test_end <= test_start:
        break
    
    # Get test data
    test_close = close.iloc[test_start:test_end]
    test_entries = entries.iloc[test_start:test_end]
    test_exits = exits.iloc[test_start:test_end]
    
    try:
        pf_test = vbt.Portfolio.from_signals(
            close=test_close, entries=test_entries, exits=test_exits,
            init_cash=100_000, fees=0.001, freq='D'
        )
        pf_hold_test = vbt.Portfolio.from_holding(test_close, init_cash=100_000, freq='D')
        
        test_return = pf_test.total_return()
        hold_return = pf_hold_test.total_return()
        test_sharpe = pf_test.sharpe_ratio()
        n_trades = pf_test.trades.count()
    except:
        continue
    
    wf_results.append({
        'fold': fold,
        'test_start': close.index[test_start].strftime('%Y-%m-%d'),
        'test_end': close.index[test_end-1].strftime('%Y-%m-%d'),
        'n_trades': n_trades,
        'test_return': test_return,
        'hold_return': hold_return,
        'excess_return': test_return - hold_return,
        'beat_hold': test_return > hold_return
    })
    
    status = '✓' if test_return > hold_return else '✗'
    trades_str = f"{n_trades} trades" if n_trades > 0 else "no trades"
    print(f"Fold {fold}: {close.index[test_start].strftime('%Y-%m')} | "
          f"{trades_str} | "
          f"Strat: {test_return*100:+.1f}% | "
          f"B&H: {hold_return*100:+.1f}% | {status}")

wf_df = pd.DataFrame(wf_results)
print(f"\nCompleted {len(wf_df)} folds")

In [ ]:
# Walk-forward summary
if len(wf_df) > 0:
    # Only count periods where there were trades
    wf_with_trades = wf_df[wf_df['n_trades'] > 0]
    
    print("\n" + "="*60)
    print("WALK-FORWARD RESULTS")
    print("="*60)
    print(f"\n{'Metric':<35} {'Value':>15}")
    print("-"*55)
    print(f"{'Total Folds':<35} {len(wf_df):>15}")
    print(f"{'Folds with Trades':<35} {len(wf_with_trades):>15}")
    print(f"{'Avg Strategy Return':<35} {wf_df['test_return'].mean()*100:>14.1f}%")
    print(f"{'Avg Buy&Hold Return':<35} {wf_df['hold_return'].mean()*100:>14.1f}%")
    print(f"{'Avg Excess Return':<35} {wf_df['excess_return'].mean()*100:>+14.1f}%")
    print(f"{'Beat Buy&Hold (all folds)':<35} {wf_df['beat_hold'].mean()*100:>14.1f}%")
    
    if len(wf_with_trades) > 0:
        print(f"{'Beat Buy&Hold (folds w/ trades)':<35} {wf_with_trades['beat_hold'].mean()*100:>14.1f}%")

In [ ]:
# Visualize
if len(wf_df) > 0:
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['Strategy vs B&H Returns', 'Cumulative Excess Return'])

    # Returns comparison
    fig.add_trace(go.Bar(x=wf_df['fold'], y=wf_df['test_return']*100, name='Strategy'), row=1, col=1)
    fig.add_trace(go.Scatter(x=wf_df['fold'], y=wf_df['hold_return']*100,
                             mode='markers', marker=dict(size=10, symbol='diamond', color='red'),
                             name='B&H'), row=1, col=1)

    # Cumulative excess
    cum_excess = (1 + wf_df['excess_return']).cumprod() - 1
    fig.add_trace(go.Scatter(x=wf_df['fold'], y=cum_excess*100,
                             mode='lines+markers', name='Cum Excess'), row=1, col=2)
    fig.add_hline(y=0, line_dash='dash', row=1, col=2)

    fig.update_layout(height=400)
    fig.show()

---
## 7. Final Summary

In [ ]:
print("\n" + "="*70)
print("DOUBLE CAPITULATION SIGNAL - FINAL RESULTS")
print("="*70)

print(f"\n📊 SIGNAL")
print(f"   Buy when: SOPR < 1 AND STH SOPR < 1")
print(f"   Logic: Both market and short-term holders selling at a loss")

print(f"\n📈 REGRESSION ANALYSIS")
print(f"   Coefficient: {coef*100:+.2f}%")
print(f"   p-value: {pval:.4f}")
print(f"   Significant: {'✓ Yes' if pval < 0.05 else '✗ No'}")

print(f"\n📊 IN-SAMPLE BACKTEST")
print(f"   Strategy Return: {pf.total_return()*100:.1f}%")
print(f"   Buy & Hold Return: {pf_hold.total_return()*100:.1f}%")
print(f"   Excess: {excess*100:+.1f}%")

if len(wf_df) > 0:
    print(f"\n🔍 WALK-FORWARD VALIDATION")
    print(f"   Folds: {len(wf_df)}")
    print(f"   Beat Buy&Hold: {wf_df['beat_hold'].mean()*100:.0f}%")
    
    if wf_df['beat_hold'].mean() > 0.5:
        verdict = "✓ SIGNAL WORKS - Use it!"
    elif wf_df['beat_hold'].mean() > 0.4:
        verdict = "~ MARGINAL - Needs refinement"
    else:
        verdict = "✗ SIGNAL DOESN'T ADD VALUE"
    
    print(f"\n🎯 VERDICT: {verdict}")

print("\n" + "="*70)

In [ ]:
# Save results
import json

results_summary = {
    'signal': 'double_capitulation',
    'condition': 'SOPR < 1 AND STH_SOPR < 1',
    'data_start': '2019+',
    'regression_coef': float(coef),
    'regression_pval': float(pval),
    'in_sample_return': float(pf.total_return()),
    'in_sample_excess': float(excess),
    'oos_return': float(wf_df['test_return'].mean()) if len(wf_df) > 0 else None,
    'beat_hold_pct': float(wf_df['beat_hold'].mean()) if len(wf_df) > 0 else None,
    'n_folds': len(wf_df),
    'signal_frequency': float(df['double_capitulation'].mean())
}

with open('../data/sopr_double_cap_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Results saved to ../data/sopr_double_cap_results.json")
print(json.dumps(results_summary, indent=2))